# AI-Enhanced Collaborative Drawing Game - Machine Learning Module
## Sketch Category Classification using K-Nearest Neighbors (KNN)

This notebook covers the complete end-to-end Machine Learning pipeline for real-time sketch classification:
1. **Dataset Overview & Category Mapping**: Mapping QuickDraw-style sketches into 5 target categories: **Animal**, **Object**, **Food**, **Vehicle**, and **Nature**.
2. **Feature Engineering**: Extracting geometric dimensions, stroke dynamics, curvature, orientation histograms, and downsampled spatial raster features.
3. **Model Training & Hyperparameter Tuning**: Tuning $k$ values ($k \in [3, 5, 7, 9, 11]$), distance metrics (`euclidean`, `manhattan`, `cosine`), and neighbor weighting schemes using Stratified Cross-Validation.
4. **Evaluation & Diagnostics**: Accuracy, Macro/Weighted F1-score, Precision, Recall, and Confusion Matrix visualization.
5. **Artifact Export**: Saving the trained KNN classifier, scaler, and label metadata using `joblib` for production deployment in FastAPI.

In [ ]:
import os
import sys
import math
import json
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Add current directory to path for local module imports
sys.path.append(os.path.abspath('.'))
from feature_extraction import CATEGORY_MAP, CATEGORY_TO_ID, extract_features_from_strokes, rasterize_strokes
from generate_dataset import generate_dataset

print("Categories Mapping:", CATEGORY_MAP)

### Step 1: Dataset Generation & Feature Extraction
Each sketch is represented as a sequence of vector strokes $[[[x_1, x_2, \dots], [y_1, y_2, \dots]], \dots]$.
We extract 77 high-information features:
- **15 Geometric & Bounding Box Features**: stroke count, total point count, total path length, average stroke length, aspect ratio, bounding area, diagonal, extent, and normalized center of mass.
- **8-bin Orientation Histogram**: captures predominant drawing stroke angles ($0$ to $2\pi$).
- **5 Curvature & Density Metrics**: mean curvature, standard deviation of curvature, overall pixel density, horizontal symmetry, and vertical symmetry.
- **49 Spatial Cell Density Features**: 28x28 rasterized grid pooled into 7x7 spatial blocks.

In [ ]:
# Load or generate dataset
dataset_path = os.path.join("dataset", "quickdraw_sampled.npz")
if os.path.exists(dataset_path):
    print(f"Loading dataset from {dataset_path}...")
    data = np.load(dataset_path)
    X, y, bitmaps = data["X"], data["y"], data["bitmaps"]
else:
    print("Generating balanced dataset across 5 categories...")
    X, y, bitmaps, meta = generate_dataset(samples_per_category=500, seed=42)
    os.makedirs("dataset", exist_ok=True)
    np.savez_compressed(dataset_path, X=X, y=y, bitmaps=bitmaps)

print(f"Dataset loaded: X shape = {X.shape}, y shape = {y.shape}, bitmaps shape = {bitmaps.shape}")

### Step 2: Exploratory Data Analysis & Sample Visualization

In [ ]:
# Visualize sample 28x28 sketches from each category
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for cid in range(5):
    idx = np.where(y == cid)[0][0]
    axes[cid].imshow(bitmaps[idx], cmap='Blues')
    axes[cid].set_title(f"{CATEGORY_MAP[cid]}\n(Class {cid})")
    axes[cid].axis('off')
plt.tight_layout()
plt.show()

### Step 3: Train / Test Split and Feature Scaling
We use an 80/20 stratified split to preserve class balance, followed by `StandardScaler` to normalize feature variances across geometric lengths, counts, and spatial cell intensities.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape[0]} samples")
print(f"Test set:     {X_test_scaled.shape[0]} samples")

### Step 4: Hyperparameter Tuning for KNeighborsClassifier
We perform a 5-fold Stratified Cross-Validation grid search across:
- $k \in [3, 5, 7, 9, 11]$
- Weights: `uniform`, `distance`
- Distance metrics: `euclidean`, `manhattan`, `cosine`

In [ ]:
param_grid = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "cosine"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(KNeighborsClassifier(), param_grid, cv=cv, scoring="accuracy", n_jobs=-1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

best_knn = grid_search.best_estimator_

### Step 5: Test Set Evaluation & Confusion Matrix

In [ ]:
y_pred = best_knn.predict(X_test_scaled)
y_prob = best_knn.predict_proba(X_test_scaled)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro")
rec = recall_score(y_test, y_pred, average="macro")
f1 = f1_score(y_test, y_pred, average="macro")

print(f"Test Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Macro Precision: {prec:.4f}")
print(f"Macro Recall:    {rec:.4f}")
print(f"Macro F1-Score:  {f1:.4f}\n")

class_names = [CATEGORY_MAP[i] for i in range(len(CATEGORY_MAP))]
print(classification_report(y_test, y_pred, target_names=class_names))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("KNN Drawing Category Classification - Confusion Matrix")
plt.xlabel("Predicted Category")
plt.ylabel("True Category")
plt.tight_layout()
plt.show()

### Step 6: Model Export for Production API

In [ ]:
os.makedirs("saved_models", exist_ok=True)
joblib.dump(best_knn, os.path.join("saved_models", "knn_drawing_model.joblib"))
joblib.dump(scaler, os.path.join("saved_models", "scaler.joblib"))
joblib.dump(CATEGORY_MAP, os.path.join("saved_models", "label_encoder.joblib"))
print("Model, Scaler, and Label Encoder saved to saved_models/")